# Mensch-in-der-Schleife: Vor-Aktions-Gates, Risikostufung und Audit-Protokollierung

Das README zu dieser Lektion führt Mensch-in-der-Schleife mit einem kurzen Ausschnitt ein, der den Benutzer nach der Antwort des Agenten zur Eingabe von `APPROVE` oder `REJECT` auffordert. Dieses Muster ist ein guter Ausgangspunkt, aber produktive HITL-Implementierungen benötigen üblicherweise zusätzlich drei weitere Komponenten:

1. Ein **Vor-Aktions-Gate**, das **vor** der Ausführung eines risikoreichen Schritts durch den Agenten läuft, damit Kosten, Unumkehrbarkeit und Latenz unter Kontrolle bleiben.
2. **Risikostufung**, sodass Aktionen mit geringem Risiko automatisch ausgeführt werden, Aktionen mit mittlerem Risiko gesammelt genehmigt werden und nur Aktionen mit hohem Risiko einen Mensch blockieren.
3. Ein **Audit-Log plus Revisionsschleife**, sodass jede Gate-Entscheidung als JSONL aufgezeichnet wird und eine Ablehnung den Agenten mit einem strukturierten Grund neu auffordert, anstatt nur `Revising...` anzuzeigen.

Dieses Notebook baut jede dieser Komponenten auf denselben Primitiven wie `06-system-message-framework.ipynb` auf. Es läuft vollständig in `DEMO_MODE = True` (keine interaktive Eingabe nötig) oder mit echten `input()`-Eingabeaufforderungen, wenn `DEMO_MODE = False`. Hinweis: Im DEMO_MODE ist der dritte Ziel-Wiederholungsversuch scriptgesteuert, damit die Schleifenmechanik vollständig sichtbar ist. Eine echte revisionsgesteuerte Neuklassifikation erfordert `DEMO_MODE = False` und eine Bedienperson.

**Nicht im Umfang enthalten (in anderen Lektionen behandelt):** Authentifizierung und Zugriffskontrolle (Lektion 06 README Bedrohung 2), Middleware für Tool-Aufrufe (Lektion 14 MAF Deep Dive), Multi-Agenten-Debattenmuster.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Muster 1: Pre-Action-Gate

Das HITL-Snippet im README ruft zuerst den Agenten auf und bittet dann den Benutzer, die Ausgabe zu genehmigen. Das ist ein **Post-Action**-Ablauf. Der Agent hat bereits ausgeführt, sodass die LLM-Kosten bereits bezahlt sind und jede Nebenwirkung (gesendete E-Mail, geschriebene Datenbankzeile, geposteter Kommentar) bereits eingetreten ist.

Ein **Pre-Action**-Ablauf fügt das Gate vor dem Ausführen des riskanten Schritts durch den Agenten ein. Der Agent schlägt die Aktion vor, das Gate entscheidet, ob ausgeführt wird, und erst bei Genehmigung tritt die Nebenwirkung auf.

| Aspekt | Genehmigung nach der Aktion (README-Snippet) | Pre-Action-Gate (dieses Notebook) |
|---|---|---|
| Wann erfolgt die Genehmigung? | Nachdem der Agent eine Ausgabe erzeugt hat | Bevor eine Nebenwirkung ausgeführt wird |
| LLM-Kosten bei Ablehnung | Bereits bezahlt | Nur für den Vorschlag, nicht für die Aktion bezahlt |
| Irreversible Nebenwirkungen | Möglich (die Aktion hat bereits stattgefunden) | Verhindert |
| Prüfungstransparenz | Genehmigung ist eine Druckanweisung | Genehmigung ist ein JSON-Datensatz mit Zeitstempel, Aktion, Grund |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Muster 2: Risikostufung

Nicht jede Aktion benötigt eine menschliche Genehmigung. Ein schreibgeschützter Nachschlag bei einer öffentlichen API hat andere Risiken als das Versenden einer Kunden-E-Mail. Beide gleich zu behandeln, verschwendet die Aufmerksamkeit des Bedieners und verlangsamt den Agenten.

Ein einfaches 3-Stufen-Modell:

| Stufe | Beispiele | Genehmigungsablauf |
|---|---|---|
| `niedrig` (schreibgeschützt) | Nachschlagen in einer Wissensdatenbank, Flugoptionen prüfen, eine öffentliche Webseite abfragen | Automatische Ausführung, protokolliert zur Prüfung |
| `mittel` (geringfügige Änderungen) | Ergebnis zwischenspeichern, Zähler erhöhen, Erinnerung planen | Automatische Ausführung, aber tägliche gebündelte Überprüfung |
| `hoch` (extern sichtbar oder irreversibel) | E-Mail senden, Karte belasten, in öffentlichen Kanal posten | Blockiert bis zur menschlichen Genehmigung |

Dies ist eine Form der Stufung. Produktive Systeme verwenden oft granulare Stufen (z. B. AWS IAM-Berechtigungsstufen, rollenbasierte Zugriffsstufen). Die unten stehende 3-Stufen-Version ist die kleinste sinnvolle Version für einen Agenten, der schreibgeschützte und nebenwirkende Aktionen mischt.

Der untenstehende Klassifikator verwendet Schlüsselwort-Heuristiken, damit die Demo deterministisch und günstig bleibt. In einem produktiven System würde man diesen durch einen gelernten Klassifikator oder eine Richtlinien-Engine ersetzen.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Muster 3: Prüfprotokoll und Überarbeitungsschleife

Ein `print("Response approved.")` ist kein Prüfprotokoll. Für Vertrauen sollte jede Tor-Entscheidung als strukturiertes Ereignis aufgezeichnet werden, das Sie später abfragen, erneut abspielen oder an eine Vorfallbewertung anhängen können.

Zwei Bestandteile:

1. **Nur anhängbare JSONL.** Eine Zeile pro Entscheidung mit Zeitstempel, Aktion, Stufe, Entscheidung, Grund. Einfach zu durchsuchen, einfach später an einen echten Protokollspeicher zu senden.
2. **Überarbeitungsschleife bei Ablehnung.** Wenn das Tor `deny` zurückgibt, fordert sich der Agent mit dem Ablehnungsgrund im Kontext selbst erneut auf, sodass der nächste Vorschlag das Problem vermeiden kann.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Zusätzliche Ressourcen

Mehrere andere öffentliche Projekte implementieren Variationen dieser HITL-Muster. Vergleichen Sie Ansätze, um herauszufinden, was zu Ihrem Stack passt:

- **LangChain** mensch-in-der-Schleife Tool-Wrapper ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): einsatzbereite Tool-Wrapper, die die Ausführung für menschliche Eingaben pausieren.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ hat dies umstrukturiert): verwendet eine Agentenrolle speziell zur Darstellung des Menschen in Multi-Agenten-Konversationen.
- **Microsoft Agent Framework (MAF)** Middleware zur Funktionsaufruf-Invocation ([docs](https://learn.microsoft.com/agent-framework/)): Middleware, die um jeden Tool-/Funktionsaufruf ausgeführt wird, geeignet für Steuerungslogik und Genehmigungsabläufe.

Jedes Projekt behandelt die drei Teilmuster unterschiedlich: LangChain verpackt sie als Tools, AutoGen verwendet eine Agentenrolle und das Microsoft Agent Framework nutzt Middleware für Funktionsaufrufe. Lesen Sie eine oder zwei Implementierungen vollständig durch, bevor Sie sich für ein Design für Ihren eigenen Agenten entscheiden.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Haftungsausschluss**:
Dieses Dokument wurde mit dem KI-Übersetzungsdienst [Co-op Translator](https://github.com/Azure/co-op-translator) übersetzt. Obwohl wir uns um Genauigkeit bemühen, beachten Sie bitte, dass automatisierte Übersetzungen Fehler oder Ungenauigkeiten enthalten können. Das Originaldokument in seiner Ursprungssprache gilt als maßgebliche Quelle. Bei kritischen Informationen wird eine professionelle menschliche Übersetzung empfohlen. Wir übernehmen keine Haftung für Missverständnisse oder Fehlinterpretationen, die aus der Verwendung dieser Übersetzung entstehen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
